In [1]:
import pandas as pd
import os
import glob
import numpy as np
import random

In [2]:
blood_df = pd.read_csv(r'E:\4thYearProjectCoding\dataset\lung_blood_markers.csv')
radiology_df = pd.read_csv(r'E:\4thYearProjectCoding\dataset\lung_clinical_reports.csv')

In [3]:
radiology_df.sample(10)

,Report_ID,Patient_Age,Patient_Sex,Clinical_Observation,Diagnosis_Label
4003,REP_04004,72,Female,Clinical indicators: tachypnea during examinat...,Viral Pneumonia
7639,REP_07640,76,Female,NAN,Tuberculosis
8662,REP_08663,60,Male,NAN,Bacterial
8230,REP_08231,53,Female,Chest CT is almost clear except for thick yell...,Corona Virus Disease
1745,REP_01746,22,Male,No suspicious cavitary lesion. Clinical indica...,Viral Pneumonia
5912,REP_05913,87,Female,"Patient reports no current cough, transient ch...",Healthy
6861,REP_06862,48,Female,"Patient reports no current cough, no fever. No...",Healthy
1159,REP_01160,50,Female,"clinical course subacute, procalcitonin not cl...",Tuberculosis
1561,REP_01562,39,Female,"symptoms for 17 days, tachypnea during examina...",Corona Virus Disease
7216,REP_07217,67,Male,NAN,Tuberculosis


In [4]:
len(blood_df), len(radiology_df)

(8900, 8900)

In [5]:
blood_df["Diagnosis_Label"] = blood_df["Diagnosis_Label"].replace(
    "COVID-19",
    "Corona Virus Disease"
)

In [6]:
blood_df.sample(10)

,Record_ID,WBC (x10^9/L),NEUT%,LYMP%,NLR,CRP (mg/L),PCT (ng/mL),Diagnosis_Label
2359,BLD02360,10.18,84.11,12.09,6.96,79.82,0.25,Tuberculosis
2007,BLD02008,5.57,73.11,18.39,3.98,27.76,0.01,Viral Pneumonia
7791,BLD07792,7.30,43.38,24.07,1.80,6.83,0.08,Healthy
8241,BLD08242,9.69,57.61,12.73,4.53,85.01,0.21,Corona Virus Disease
7131,BLD07132,5.31,50.96,25.94,1.96,2.84,0.06,Healthy
4685,BLD04686,5.76,48.89,36.74,1.33,2.68,0.05,Healthy
5783,BLD05784,22.76,85.43,11.57,7.38,102.86,3.13,Bacterial
7941,BLD07942,5.98,65.97,19.89,3.32,28.25,0.26,Tuberculosis
6467,BLD06468,6.04,52.72,38.52,1.37,2.51,0.02,Healthy
3727,BLD03728,14.41,75.97,10.55,7.20,9.15,0.15,Tuberculosis


In [7]:
label_map = {
    "Healthy":                 "Normal",
    "Viral Pneumonia":         "Viral Pneumonia",
    "Bacterial":     "Bacterial Pneumonia",
    "Corona Virus Disease":"Corona Virus Disease",
    "Tuberculosis":            "Tuberculosis",
}

In [8]:
blood_df["Diagnosis_Label"]    = blood_df["Diagnosis_Label"].map(label_map)
radiology_df["Diagnosis_Label"] = radiology_df["Diagnosis_Label"].map(label_map)

In [14]:
TRAIN_DIR = r"E:\4thYearProjectCoding\images"

image_pool = {}   # { label: [path, path, ...] }
for label_folder in os.listdir(TRAIN_DIR):
    folder_path = os.path.join(TRAIN_DIR, label_folder)
    if not os.path.isdir(folder_path):
        continue
    images = glob.glob(os.path.join(folder_path, "*"))
    image_pool[label_folder] = images

In [15]:
blood_by_label     = {lbl: grp.reset_index(drop=True)
                      for lbl, grp in blood_df.groupby("Diagnosis_Label")}
radiology_by_label = {lbl: grp.reset_index(drop=True)
                      for lbl, grp in radiology_df.groupby("Diagnosis_Label")}

In [16]:
rows = []

all_labels = set(image_pool) & set(blood_by_label) & set(radiology_by_label)

for label in all_labels:
    images     = image_pool[label]
    blood_rows = blood_by_label[label]
    radio_rows = radiology_by_label[label]

    n = min(len(images), len(blood_rows), len(radio_rows))

    # Shuffle so sampling is random each run
    images_sample = random.sample(images, n)
    blood_sample  = blood_rows.sample(n).reset_index(drop=True)
    radio_sample  = radio_rows.sample(n).reset_index(drop=True)

    for i in range(n):
        rows.append({
            "patient_no":    blood_sample.loc[i, "Record_ID"],
            "image_path":    images_sample[i],
            "Patient_Age":   radio_sample.loc[i, "Patient_Age"],
            "Patient_Sex":   radio_sample.loc[i, "Patient_Sex"],
            "WBC (x10^9/L)": blood_sample.loc[i, "WBC (x10^9/L)"],
            "NEUT%":         blood_sample.loc[i, "NEUT%"],
            "LYMP%":         blood_sample.loc[i, "LYMP%"],
            "NLR":           blood_sample.loc[i, "NLR"],
            "CRP (mg/L)":    blood_sample.loc[i, "CRP (mg/L)"],
            "PCT (ng/mL)":   blood_sample.loc[i, "PCT (ng/mL)"],
            "Clinical_Observation":radio_sample.loc[i, "Clinical_Observation"],
            "label":         label,
        })


In [17]:
df = pd.DataFrame(rows)
unified_df = df.sample(frac=1).reset_index(drop=True)
unified_df.to_csv("unified_dataset_new3.csv", index=False)
print(f"Saved {len(unified_df)} rows → unified_dataset_new3.csv")
print(unified_df["label"].value_counts())

Saved 8900 rows → unified_dataset_new3.csv
label
Corona Virus Disease    1780
Bacterial Pneumonia     1780
Tuberculosis            1780
Viral Pneumonia         1780
Normal                  1780
Name: count, dtype: int64


In [18]:
print(df["image_path"].head(5))

0    E:\4thYearProjectCoding\images\Normal\IM-0438-...
1    E:\4thYearProjectCoding\images\Normal\test_0_5...
2    E:\4thYearProjectCoding\images\Normal\NORMAL2-...
3    E:\4thYearProjectCoding\images\Normal\NORMAL2-...
4    E:\4thYearProjectCoding\images\Normal\NORMAL2-...
Name: image_path, dtype: str


In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("unified_dataset_new3.csv")

num_cols = ["Patient_Age", "WBC (x10^9/L)", "NEUT%", "LYMP%", 
            "NLR", "CRP (mg/L)", "PCT (ng/mL)"]

print("=== NaN counts per column ===")
print(df[num_cols].isna().sum())

print("\n=== Inf counts per column ===")
print(np.isinf(df[num_cols].values.astype(float)).sum(axis=0))

print(f"\nTotal rows: {len(df)}")
print(f"Rows with ANY NaN in lab cols: {df[num_cols].isna().any(axis=1).sum()}")

=== NaN counts per column ===
Patient_Age      0
WBC (x10^9/L)    0
NEUT%            0
LYMP%            0
NLR              0
CRP (mg/L)       0
PCT (ng/mL)      0
dtype: int64

=== Inf counts per column ===
[0 0 0 0 0 0 0]

Total rows: 8900
Rows with ANY NaN in lab cols: 0


In [20]:
import pandas as pd
df = pd.read_csv("unified_dataset_new3.csv")
print(df["label"].value_counts())

label
Corona Virus Disease    1780
Bacterial Pneumonia     1780
Tuberculosis            1780
Viral Pneumonia         1780
Normal                  1780
Name: count, dtype: int64
